# 第1章 Python の基本 ―― 研究のパラメータを書いてみる

**この章でできるようになること**
- 変数・数・文字列・リスト・タプル・辞書を使える
- `for` と `if` で「ステップの電圧 V = nΩ₁ + mΩ₂」を一覧にできる
- 関数を作って、同じ計算を何度も使い回せる
- 小数の計算に小さな誤差があることを知り、正しく比べられる

**例え：** Python のプログラムは「料理のレシピ」です。変数は材料を入れておく**ラベル付きの容器**、関数は「みじん切り」のような**決まった手順に名前を付けたもの**、`for` は「玉ねぎ5個を同じ手順で切る」の**くり返し**です。

| 用語 | 意味 | 例え |
|---|---|---|
| 変数 | 値に名前を付けて覚えておくもの | ラベルを貼った容器 |
| 型（type） | 値の種類（整数 `int`、小数 `float`、文字列 `str` など） | 容器に入っているのが水か粉か |
| リスト（list） | 順番のある値の並び。あとから変えられる | 付箋を並べて貼ったボード |
| タプル（tuple） | 順番のある値の並び。あとから変えられない | ラミネートした名札 |
| 辞書（dict） | 「名前 → 値」の対応表 | 電話帳 |
| 関数 | 入力を受け取り、決まった手順で結果を返すもの | 自動販売機 |
| 浮動小数点数 | コンピュータの小数。2進数で表すので小さな誤差が出る | 1/3 を 0.333 と書くときの切り捨て |

> 研究コードとのつながり：この章の内容は `shapiro-research/py/load.py` の `STEPS` と `O1,O2`、`py/coherent.py` の `for` 文で使われています。

In [1]:
# 準備：教材の部品（kyozai/）を import できるようにする。どの章でも最初に1回だけ実行する。
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "kyozai").exists():   # kyozai/ があるフォルダまで上にたどる
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("教材の場所:", ROOT.name)

教材の場所: python-from-my-research


## 1.1 変数と型

研究の標準パラメータを変数に入れてみます。`#` から後ろはコメント（メモ）で、実行されません。

In [2]:
beta = 10.0      # 減衰 β
O1 = 0.5         # 1つ目の交流の周波数 Ω₁
O2 = 0.8         # 2つ目の交流の周波数 Ω₂
n_points = 33    # 振幅 A を何点とるか
name = "two-tone Shapiro"

print(beta, O1, O2, n_points, name)
print(type(beta), type(n_points), type(name))

10.0 0.5 0.8 33 two-tone Shapiro
<class 'float'> <class 'int'> <class 'str'>


`10.0` は小数（`float`）、`33` は整数（`int`）です。MATLAB では数はふつう全部 `double`（小数）ですが、Python では**小数点を書くかどうかで型が変わります**。

文字列の中に変数の値を埋め込むには **f-string**（`f"..."`）を使います。`{beta:.1f}` は「小数点以下1桁で書く」という指定です。研究コードの `print(f'{b:5.1f} ...')` も同じ書き方です。

In [3]:
print(f"β = {beta:.1f}, Ω1 = {O1}, Ω2 = {O2}")
print(f"周波数の比 Ω1/Ω2 = {O1/O2:.4f}")

β = 10.0, Ω1 = 0.5, Ω2 = 0.8
周波数の比 Ω1/Ω2 = 0.6250


## 1.2 リスト・タプル・辞書

研究で扱う5つのステップ (n, m) を並べます。`(0, 1)` のような丸かっこは**タプル**、全体を囲む角かっこ `[...]` は**リスト**です。

In [4]:
steps = [(0, 1), (1, 0), (-1, 1), (1, 1), (2, 0)]
print("ステップの数:", len(steps))
print("最初:", steps[0])      # Python は 0 から数える（MATLAB は 1 から）
print("最後:", steps[-1])     # -1 は「後ろから1つ目」
print("2番目から3つ:", steps[1:4])   # 1, 2, 3 番目（4 は含まない）

ステップの数: 5
最初: (0, 1)
最後: (2, 0)
2番目から3つ: [(1, 0), (-1, 1), (1, 1)]


**MATLAB との違い（大事）**：Python の番号は **0 始まり**、範囲 `a:b` は **b を含まない**。`steps[1:4]` は「1, 2, 3 番目」です。

辞書は「名前 → 値」の表です。パラメータをひとまとめにするときに便利です。

In [5]:
params = {"beta": 10.0, "O1": 0.5, "O2": 0.8, "dt": 1/32}
print(params["beta"])
params["N_avg"] = 20106      # あとから足せる
for key, value in params.items():
    print(f"{key:>6s} = {value}")

10.0
  beta = 10.0
    O1 = 0.5
    O2 = 0.8
    dt = 0.03125
 N_avg = 20106


## 1.3 for 文と if 文 ―― ステップの電圧を一覧にする

ステップは電圧 **V = nΩ₁ + mΩ₂** に出ます。`for` で5つを順に計算します。Python では **字下げ（インデント）がブロックの範囲** を表します（MATLAB の `end` の代わり）。

In [6]:
for n, m in steps:              # タプルを n と m に分けて受け取る
    V = n * O1 + m * O2
    print(f"(n,m) = ({n:2d},{m:2d})  →  V = {V:.2f}")

(n,m) = ( 0, 1)  →  V = 0.80
(n,m) = ( 1, 0)  →  V = 0.50
(n,m) = (-1, 1)  →  V = 0.30
(n,m) = ( 1, 1)  →  V = 1.30
(n,m) = ( 2, 0)  →  V = 1.00


`enumerate` を使うと「何番目か」も一緒にもらえます。研究コードの `for s,(n,m,V,sr) in enumerate(STEPS):` はこの形です。

In [7]:
for s, (n, m) in enumerate(steps):
    V = n * O1 + m * O2
    if V < 0.5:
        size = "低い電圧"
    elif V < 1.0:
        size = "中くらい"
    else:
        size = "高い電圧"
    print(s, (n, m), round(V, 2), size)

0 (0, 1) 0.8 中くらい
1 (1, 0) 0.5 中くらい
2 (-1, 1) 0.3 低い電圧
3 (1, 1) 1.3 高い電圧
4 (2, 0) 1.0 高い電圧


## 1.4 関数を作る

同じ計算を何度も書かずにすむように、**関数**にします。`O1=0.5` のように書くと、呼ぶときに省略できる「既定値」になります。

In [8]:
def step_voltage(n, m, O1=0.5, O2=0.8):
    """ステップ (n, m) の電圧 V = nΩ1 + mΩ2 を返す。"""
    return n * O1 + m * O2

print(step_voltage(-1, 1))            # 既定の Ω1=0.5, Ω2=0.8
print(step_voltage(-1, 1, O2=0.81))   # Ω2 だけ変える

0.30000000000000004
0.31000000000000005


`"""..."""` は **docstring**（関数の説明書き）です。`help(step_voltage)` で読めます。

## 1.5 小数の誤差に気をつける

`step_voltage(-1, 1)` は 0.3 のはずなのに、`0.30000000000000004` と出ました。コンピュータは小数を2進数で持つので、0.1 や 0.3 をぴったり表せません。**小数どうしを `==` で比べてはいけません**。

In [9]:
V = step_voltage(-1, 1)
print(V == 0.3)                 # False になる
print(abs(V - 0.3) < 1e-9)      # 「十分近いか」で比べる → True

import math
print(math.isclose(V, 0.3))     # 同じことをする標準の関数

False
True
True


## 1.6 リスト内包表記 ―― 同じ電圧に届く (n, m) を探す

`[式 for ... in ... if 条件]` と書くと、条件に合うものだけを集めたリストを1行で作れます。

-10 ≤ n, m ≤ 10 の範囲で、**V = 0.3 に届く (n, m) をすべて**探してみます。

In [10]:
same_V = [(n, m) for n in range(-10, 11) for m in range(-10, 11)
          if math.isclose(step_voltage(n, m), 0.3, abs_tol=1e-9)]
print(same_V)

[(-9, 6), (-1, 1), (7, -4)]


(-1, 1) のほかに (7, -4) や (-9, 6) も見つかりました。並びを見ると **n は 8 ずつ、m は 5 ずつ**変わっています。Ω₁:Ω₂ = 0.5:0.8 = 5:8 なので 8Ω₁ = 5Ω₂（= 4）となり、n を 8 増やして m を 5 減らしても電圧が変わらないからです。

この「同じ電圧にいくつもの (n, m) が届く」ことが、研究報告書（`報告_(-1,1)第3ローブの原因_20261102.md`）の話の出発点です。第9章でもう一度使います。

**例え：** 5拍子と8拍子の太鼓は、40拍ごとに頭がそろいます。リズムがそろうので、「同じ音の高さ（電圧）」を別の叩き方の組み合わせでも鳴らせます。

## 練習問題

1. Ω₂ = 0.81 にすると、-10 ≤ n, m ≤ 10 で V = −0.5 + 0.81 = 0.31 に届く (n, m) は何組ありますか。
2. 研究コードの `STEPS` は `(n, m, V, srange)` の4つ組です。`steps` から `V` を計算して、`(n, m, V)` の3つ組のリストを内包表記で作ってください。

<details><summary>答え（クリックで開く）</summary>

```python
# 1.
r = [(n, m) for n in range(-10, 11) for m in range(-10, 11)
     if math.isclose(step_voltage(n, m, O2=0.81), 0.31, abs_tol=1e-9)]
print(r)   # [(-1, 1)] の1組だけ。0.5:0.81 は小さな整数の比にならないので、ほかの組は届かない
# 2.
triples = [(n, m, step_voltage(n, m)) for n, m in steps]
```
</details>

In [11]:
# 練習1 の答えを実際に確かめる
r = [(n, m) for n in range(-10, 11) for m in range(-10, 11)
     if math.isclose(step_voltage(n, m, O2=0.81), 0.31, abs_tol=1e-9)]
print(r)

[(-1, 1)]


## まとめ
- 変数は容器、関数は決まった手順。Python の番号は 0 始まりで、`a:b` は b を含まない。
- 小数は `==` で比べず、`math.isclose` などで「十分近いか」を見る。
- Ω₁:Ω₂ = 5:8 では、(n, m) と (n+8, m−5) が同じ電圧に届く。

次の章では、たくさんの数をまとめて計算する **NumPy** を使います。